# 4.2 — ガイド付きプロジェクト：学習センター分析

**具体的な問い：** 2026年1〜6月について、Python FoundationsとDigital Skillsのどちらが全体修了率が高く、どちらが一人修了当たり教材費が低いでしょうか。差を数値で示し、この24件だけでコースの優劣を決められるか判断し、次に必要なデータを一つ提案します。

## 1. 使用するファイルと完成条件

ファイル名は`learning-centres-practice.csv`です。Moodleの「4.2 データセットとプロジェクト手順」から直接ダウンロードできます。Python Labでは`/home/jovyan/work/data/learning-centres-practice.csv`にあり、このNotebookの次のセルが開始フォルダに依存せず探します。

読み込み直後は**24行×10列**です。対象コースは`Python Foundations`と`Digital Skills`、対象月は`2026-01`〜`2026-06`です。最終的に、品質監査表、コース別集計表、照合結果、主図1点、300〜500字の回答をこのNotebookへ残します。

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

def find_course_data(filename):
    roots=[Path.cwd(),*Path.cwd().parents,Path.home()/"work",Path("/opt/python-lab/course-materials")]
    checked=[]
    for root in roots:
        for p in (root/"data"/filename,root/filename):
            if p in checked: continue
            checked.append(p)
            if p.is_file(): return p
    raise FileNotFoundError("Course data was not found:\n"+"\n".join(map(str,checked)))

data_file=find_course_data("learning-centres-practice.csv")
raw=pd.read_csv(data_file)
print("Source:",data_file.resolve(),"shape:",raw.shape)


## ファイルを開けたことを確認する

次の出力で、実際に読んだ絶対パス、`(24, 10)`、10列の名前、先頭5行を確認します。異なる形なら、そのまま進まずファイルを確認します。

In [ ]:
print(data_file.resolve())
print("Shape:", raw.shape)
print("Columns:", raw.columns.tolist())
display(raw.head())
assert raw.shape == (24, 10)


## 2. 次の10列を検査する

`month`, `centre_id`, `centre_name`, `district`, `course`, `registered`, `attended`, `completed`, `training_hours`, `material_cost`が存在することを確認します。次に型、列別欠損件数、地区名一覧、人数・時間・費用の最小値と最大値を表示します。ここではまだ修正しません。

In [ ]:
required={"month","centre_id","district","course","registered","attended","completed","material_cost"}
missing=required-set(raw.columns)
if missing: raise KeyError(sorted(missing))
print(raw.dtypes); print(raw.isna().sum()); print(sorted(raw["district"].dropna().unique()))


## 3. 三つの品質検査を別々に実行する

次の規則を、名前付きブールマスクとして別々に数えます。

1. `attended`が欠損している。
2. `completed > attended`である（両方の値がある行だけ）。
3. `centre_id + month + course`が重複している。

地区名は元の`district_raw`を残して前後空白と大文字・小文字を正規化します。人数を推測で書き換えず、問題行を`analysis_ready=False`にします。監査表には問題名、規則、影響件数、処置を記録します。

In [ ]:
clean=raw.copy(); clean["district_raw"]=clean["district"]; clean["district"]=clean["district"].astype("string").str.strip().str.title()
missing_attended=clean["attended"].isna(); invalid_completion=clean["completed"].notna() & clean["attended"].notna() & (clean["completed"]>clean["attended"])
key=["centre_id","month","course"]; duplicate=clean.duplicated(key,keep=False)
clean["analysis_ready"]=~(missing_attended|invalid_completion|duplicate)
analysis=clean.loc[clean["analysis_ready"]].copy()
audit=pd.DataFrame({"issue":["missing attended","completion above attendance","duplicate business key"],"affected":[int(missing_attended.sum()),int(invalid_completion.sum()),int(duplicate.sum())],"action":["flag; exclude from rate analysis","flag; do not guess","review duplicate group"]})
audit


## 4. この7列のコース別集計表を完成させる

出力を`summary`とし、`course`, `records`, `centres`, `registered`, `completed`, `material_cost`, `completion_rate`, `cost_per_completion`を含めます。

- `completion_rate = completed合計 / registered合計 × 100`
- `cost_per_completion = material_cost合計 / completed合計`

各行の率や費用を先に計算して単純平均してはいけません。二つのコースの修了率差は**パーセントポイント**、費用差は教材費と同じ単位で計算します。

In [ ]:
summary=analysis.groupby("course").agg(records=("centre_id","size"),centres=("centre_id","nunique"),registered=("registered","sum"),completed=("completed","sum"),material_cost=("material_cost","sum")).reset_index()
summary["completion_rate"]=summary["completed"]/summary["registered"]*100
summary["cost_per_completion"]=summary["material_cost"]/summary["completed"]
summary.round(2)


## 5. 件数と合計を照合する

元件数＝分析可能件数＋フラグ件数、グループ合計＝分析用明細合計を`assert`で確認します。

In [ ]:
assert len(raw)==int(clean["analysis_ready"].sum())+int((~clean["analysis_ready"]).sum())
assert summary["registered"].sum()==analysis["registered"].sum()
assert summary["completed"].sum()==analysis["completed"].sum()
assert abs(summary["material_cost"].sum()-analysis["material_cost"].sum())<1e-9
print("Validation passed")


## 6. 一つの問いに一つの主図を作る

全体修了率を0〜100%の横棒で比較します。一人修了当たり教材費は表で併記し、必要なら第二図にします。軸、単位、タイトル、件数を明示します。

In [ ]:
plot_data=summary.sort_values("completion_rate")
fig,ax=plt.subplots(figsize=(7,4)); ax.barh(plot_data["course"],plot_data["completion_rate"],color="#356a9a"); ax.set(xlabel="Overall completion rate (%)",ylabel="Course",title=f"Completion by course (analysis-ready centre-months n={len(analysis)})"); ax.set_xlim(0,100); ax.grid(axis="x",alpha=.25); plt.tight_layout(); plt.show()


## 7. 次の5項目へ順番に答える

300〜500字程度で、次を明記します。

1. 各コースの全体修了率と、その差（パーセントポイント）。
2. 各コースの一人修了当たり教材費と、その差。
3. どちらが高い／低いか。ただし『優れている』とは書かない。
4. 分析対象行数、対象期間、少なくとも一つの限界。
5. 優劣や原因を検討するため、次に必要なデータを一つ。

In [ ]:
# ここに報告を書きます。


## 提出前チェックリスト

- 最初から最後まで「すべて実行」でエラーがない。
- 読み込んだパスと`(24, 10)`が表示される。
- 三つの品質問題が別々に数えられ、監査表がある。
- `summary`に指定した8列と2コースがある。
- 行数・登録者・修了者・教材費の照合が通る。
- 主図は0〜100%軸、タイトル、軸名、単位、分析対象件数を持つ。
- 報告文が上の5項目すべてへ回答する。